# Modèle français + kirundi (base : Pythia-14m)

Ce notebook enchaîne **tout** le projet, dans l'ordre :

| Étape | Ce qui se passe | Durée (GPU T4) |
|---|---|---|
| 0 | Préparer Colab : GPU, Google Drive, code | 2 min |
| 1 | Générer toutes les données (conjugaison, accords, Bible, cours...) | 10 à 20 min |
| 2 | Créer le tokenizer français + kirundi | 5 min |
| 3 | **Phase A** : bases de la langue sur le petit modèle (14M) | 1 à 2 h |
| 4 | **Agrandir** le modèle : 14M -> 122M (`grow.py`) | 1 min |
| 5 | **Phase B** : le grand modèle apprend tout | 6 à 10 h |
| 6 | **Phase D** : apprendre à répondre (questions / réponses, traduction) | 1 à 2 h |
| 7 | **Tests** : notes par catégorie + vitesse | 10 min |
| 8 | Essayer le modèle soi-même, le sauvegarder | |

**Si Colab coupe** (ça arrive après quelques heures) : relance les cellules de l'étape 0, puis la cellule de la phase en cours. `train.py` **reprend tout seul** au dernier point sauvegardé sur Google Drive. Les étapes déjà finies sont sautées automatiquement.

**Avant de commencer** : glisse le CONTENU du dossier `A_METTRE_DANS_DRIVE` (préparé sur ton ordinateur)
à la racine de **Mon Drive** (compte devbienvenu@gmail.com). Tu dois avoir :
- `Mon Drive/sources/` (les deux bases de la Bible)
- `Mon Drive/donnees/` (toutes les données, déjà générées)
- `Mon Drive/modele/tokenizer_fr_rn/` (le tokenizer, déjà créé)

Les étapes 1 et 2 sont alors **sautées automatiquement** : tu commences directement à l'entraînement (étape 3).

**Suivi à distance** : chaque étape écrit son journal dans `MyDrive/modele/journaux/` (données, phases A/B/D, notes des tests). Claude les lit dans ton Drive pour décider des corrections.

## Étape 0 : préparer Colab

In [ ]:
# 0.1  Vérifier le GPU (menu : Exécution > Modifier le type d'exécution > GPU T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 0.2  Brancher Google Drive : TOUT est sauvegardé là (données, modèles)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 0.3  Récupérer le code depuis GitHub
#      Ton dépôt : NiBienvenu/modele_train.
#      Dépôt privé : mets un token dans les Secrets de Colab (icône clé) sous le nom GH_TOKEN.
DEPOT = "https://github.com/NiBienvenu/modele_train.git"

import os
if not os.path.exists("/content/modele_train"):
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        DEPOT = DEPOT.replace("https://", f"https://{token}@")
    except Exception:
        pass  # dépôt public
    !git clone -q $DEPOT /content/modele_train
else:
    !cd /content/modele_train && git pull -q
%cd /content/modele_train
!pip install -q -r requirements.txt

In [ ]:
# 0.4  Les dossiers utilisés partout (sur Google Drive)
import os

# Le code téléchargé à l'étape 0.3. Après un redémarrage de Colab, on revient
# dans /content : chaque cellule importante se replace donc ici.
CODE = "/content/modele_train"
def aller_au_code():
    if not os.path.exists(f"{CODE}/train.py"):
        raise SystemExit("Le code n'est pas dans " + CODE + " : relance la cellule 0.3 et lis son message "
                         "(dépôt privé : il faut le secret GH_TOKEN).")
    os.chdir(CODE)
aller_au_code()
print("Dossier de travail :", os.getcwd())

RACINE  = "/content/drive/MyDrive"
SOURCES = f"{RACINE}/sources"          # bible_kirundi.db, bible_francais.db
D       = f"{RACINE}/donnees"          # données générées
QR      = f"{D}/qr"                    # questions / réponses (phase D)
M       = f"{RACINE}/modele"           # modèles
for d in (D, QR, M, f"{M}/journaux"):
    os.makedirs(d, exist_ok=True)
# Vérification : tout doit être "OK" avant de continuer
for chemin in (f"{D}/conjugaison_fr", f"{D}/conjugaison_rn", f"{D}/grammaire_rn", f"{D}/bible",
               f"{D}/cours", f"{D}/lexique_rn", f"{D}/fsi_rn", f"{QR}/cours.jsonl", f"{M}/tokenizer_fr_rn"):
    print(("OK      " if os.path.exists(chemin) else "MANQUANT"), chemin)

def fait(chemin):
    """True si une étape est déjà finie (pour ne pas la refaire après une coupure)."""
    return os.path.exists(chemin)

def verifier(chemin, etape):
    """Arrête le notebook avec un message clair si une étape n'a pas produit son résultat."""
    if os.path.exists(chemin):
        print(f"OK : {etape} terminée -> {chemin}")
        return
    nom = etape.split()[-1]
    raise SystemExit(
        f"ÉCHEC : {etape} n'a pas produit {chemin}.\n"
        f"Regarde la fin du journal : {M}/journaux/{nom}.txt (ou la sortie de la cellule).\n"
        f"Ne passe PAS à l'étape suivante : corrige d'abord, puis relance cette cellule.")

def meilleur_modele():
    """Le modèle le plus avancé qui existe : phase D, sinon B, sinon A."""
    for ph in ("phaseD", "phaseB", "phaseA"):
        for sous in ("finale", "meilleure"):
            if os.path.exists(f"{M}/{ph}/{sous}/config.json"):
                return f"{M}/{ph}/{sous}"
    raise SystemExit("Aucun modèle entraîné trouvé dans " + M + " : lance d'abord la phase A (étape 3).")

In [ ]:
# 0.5  OÙ EN SUIS-JE ?  (à lancer à tout moment, surtout après une erreur ou une coupure)
etapes = [("Phase A", f"{M}/phaseA/finale"), ("Agrandissement", f"{M}/grand_init"),
          ("Phase B", f"{M}/phaseB/finale"), ("Phase D", f"{M}/phaseD/finale")]
for nom, chemin in etapes:
    reprise = os.path.exists(chemin.replace("finale", "derniere"))
    etat = "FINIE" if os.path.exists(chemin) else ("en cours (reprise possible)" if reprise else "pas commencée")
    print(f"{nom:15s} : {etat}")
print()
for ph in ("phaseA", "phaseB", "phaseD"):
    j = f"{M}/journaux/{ph}.txt"
    if os.path.exists(j):
        print(f"===== fin de {j}")
        !tail -n 15 "$j"


## Étape 1 : générer les données

Chaque script écrit des fichiers `.txt` (phase A et B) et `.jsonl` (questions / réponses, phase D).
Les éléments réservés aux **tests** (10 % des phrases traduites, 5 % des verbes) sont **exclus** automatiquement (voir `separation.py`).

In [ ]:
aller_au_code()
# 1.1  Français : conjugaison de 7 015 verbes, 12 temps
if not fait(f"{D}/conjugaison_fr"):
    # Seulement si les données manquent : outils de génération (sans casser numpy)
    !pip install -q -r requirements-donnees.txt && pip install -q --no-deps mlconjug3
    !python gen_conjugaison_fr.py --sortie $D/conjugaison_fr --sft $QR/conjugaison_fr.jsonl

In [ ]:
aller_au_code()
# 1.2  Kirundi : livre FSI (domaine public, téléchargé automatiquement)
if not fait(f"{D}/fsi_rn/fsi_rn.txt"):
    !pip install -q -r requirements-donnees.txt
    !python extraire_fsi.py --sortie $D/fsi_rn

In [ ]:
aller_au_code()
# 1.3  Bible kirundi + français (bases SQLite), passages alignés, part de 25 %
if not fait(f"{D}/bible/bible_bilingue_25.txt"):
    !python extraire_bible_db.py --rn $SOURCES/bible_kirundi.db --fr $SOURCES/bible_francais.db \
        --sortie $D/bible --sft $QR/traduction_bible.jsonl

In [ ]:
aller_au_code()
# 1.4  Lexique français <-> kirundi (ta liste de verbes)
#      On travaille sur des COPIES des listes de verbes : les fichiers du dépôt ne changent pas.
if not fait(f"{D}/lexique_rn/lexique_rn.txt"):
    !tail -n +2 $D/bible/mots_bible_rn.tsv | cut -f1 > /tmp/mots.tsv
    !cut -f1 $D/fsi_rn/mots_frequents.tsv >> /tmp/mots.tsv
    !cp data/verbes_rn.tsv /tmp/verbes_copie.tsv
    !python importer_dictionnaire.py --mots /tmp/mots.tsv --verbes /tmp/verbes_copie.tsv \
        --a_completer /tmp/a_completer.tsv --sortie $D/lexique_rn --sft $QR/lexique_rn.jsonl

In [ ]:
aller_au_code()
# 1.5  Cours de Cox : phrases traduites par une experte (+ tests mis de côté)
if not fait(f"{D}/cours/cours_bilingue.txt"):
    !python combiner_cours.py --resultats data/cours_resultats --sortie $D/cours --sft $QR/cours.jsonl

In [ ]:
aller_au_code()
# 1.6  Kirundi : conjugaison (3 478 verbes, 14 temps), accords (classes), pronoms objets, subordonnées
if not fait(f"{D}/conjugaison_rn"):
    !python gen_conjugaison_rn.py --sortie $D/conjugaison_rn --sft $QR/conjugaison_rn.jsonl
if not fait(f"{D}/grammaire_rn/accords_rn_001.txt"):
    !python gen_accords_rn.py --sortie $D/grammaire_rn --sft $QR/accords_rn.jsonl
if not fait(f"{D}/grammaire_rn/syntaxe_rn_001.txt"):
    !python gen_syntaxe_rn.py --sortie $D/grammaire_rn --sft $QR/syntaxe_rn.jsonl

In [ ]:
aller_au_code()
# 1.7  Le jeu de tests (568 questions) et le bilan des données
#      tests/tests.jsonl est déjà dans le dépôt : on ne le refait que s'il manque
#      (il faut alors mlconjug3, outil de génération, installé SANS ses dépendances).
if not fait("tests/tests.jsonl"):
    !pip install -q -r requirements-donnees.txt && pip install -q --no-deps mlconjug3
    !python construire_tests.py 2>&1 | tee $M/journaux/donnees.txt
else:
    !echo "Tests déjà prêts : $(wc -l < tests/tests.jsonl) questions dans tests/tests.jsonl" | tee $M/journaux/donnees.txt
!du -sh $D/* | sort -h | tee -a $M/journaux/donnees.txt
!wc -l $QR/*.jsonl

## Étape 2 : le tokenizer

Créé **une seule fois**, sur **toutes** les données. Ensuite, il ne doit plus jamais changer
(sinon le modèle ne reconnaît plus ses propres numéros).

In [ ]:
aller_au_code()
if not fait(f"{M}/tokenizer_fr_rn"):
    !python train_tokenizer.py --source "$D/**/*.txt=1" --taille_vocab 24000 --sortie $M/tokenizer_fr_rn

## Étape 3 : phase A (petit modèle, 14M)

Le modèle apprend les morceaux de mots et les règles simples. Surveille deux chiffres :
- la **loss** doit descendre ;
- les **phrases d'exemple** affichées à chaque évaluation doivent devenir du vrai français / kirundi.

In [ ]:
aller_au_code()
if not fait(f"{M}/phaseA/finale"):
    !python -u train.py --config configs/phaseA.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a $M/journaux/phaseA.txt
verifier(f"{M}/phaseA/finale", "phaseA")

## Étape 4 : agrandir le modèle (14M -> 122M)

Le savoir de la phase A est **gardé** : chaque tête d'attention est copiée à sa place, chaque couche est dupliquée.
Pour une autre taille, change `--hidden` / `--layers` / `--heads` (par exemple 768 / 16 / 12 = 150M, maximum conseillé 1024 / 12 / 16 = 200M).

In [ ]:
aller_au_code()
if not fait(f"{M}/grand_init"):
    !python grow.py --source $M/phaseA/finale --sortie $M/grand_init --hidden 768 --layers 12 --heads 12
verifier(f"{M}/grand_init", "agrandissement")

## Étape 5 : phase B (grand modèle)

La plus longue. **Si Colab coupe, relance l'étape 0 puis cette cellule** : l'entraînement reprend où il s'était arrêté.
Si la mémoire du GPU déborde (*CUDA out of memory*), baisse `taille_lot` et augmente `accumulation` dans `configs/phaseB.yaml` (par exemple 8 et 16).

In [ ]:
aller_au_code()
if not fait(f"{M}/phaseB/finale"):
    !python -u train.py --config configs/phaseB.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a $M/journaux/phaseB.txt
verifier(f"{M}/phaseB/finale", "phaseB")

## Étape 6 : phase D (apprendre à répondre)

In [ ]:
aller_au_code()
if not fait(f"{M}/phaseD/finale"):
    !python -u train.py --config configs/phaseD.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a $M/journaux/phaseD.txt
verifier(f"{M}/phaseD/finale", "phaseD")

## Étape 7 : les tests

Pour chaque catégorie : **exact** (% de réponses parfaites) et **chrF** (ressemblance, la mesure habituelle des traductions, de 0 à 100).
Les catégories marquées **"NON (vraie preuve)"** n'ont jamais été vues à l'entraînement : ce sont elles qui montrent si le modèle a **compris**.

Tu peux aussi tester un modèle intermédiaire (`phaseB/finale`) pour comparer.

In [ ]:
aller_au_code()
!python -u evaluer.py --modele {meilleur_modele()} --montrer 3 --sortie $M/journaux/resultats_tests.jsonl 2>&1 | tee $M/journaux/notes_tests.txt

In [ ]:
aller_au_code()
# Voir les erreurs d'une catégorie en détail
!python evaluer.py --modele {meilleur_modele()} --categorie traduction_rn_fr --montrer 15 --sortie $M/journaux/resultats_traduction.jsonl

## Étape 8 : essayer le modèle et le sauvegarder

In [ ]:
aller_au_code()
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

chemin = meilleur_modele()
print("Modèle utilisé :", chemin)
tok = AutoTokenizer.from_pretrained(chemin)
model = AutoModelForCausalLM.from_pretrained(chemin).to("cuda").eval()

def demander(question, max_tokens=120):
    ids = tok(f"<|user|> {question} <|assistant|>", return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_tokens, do_sample=False,
                             eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for q in ["Traduis en français : Ndagenda ku kazi uyu musi.",
          "Traduis en kirundi : Les enfants qui étudient savent lire.",
          "Conjugue le verbe kwandika au futur (après aujourd'hui).",
          "Réponds en kirundi avec un pronom : Urabona igitabo?",
          "Comment dit-on « remercier » en kirundi ?"]:
    print("Q :", q)
    print("R :", demander(q), "\n")

In [ ]:
# Pose tes propres questions
print(demander("Traduis en français : Umwana arakora cane."))

In [ ]:
# Sauvegarde : un fichier .zip du modèle final (pour ton projet, Docker, etc.)
dossier_modele = meilleur_modele()
!cd "$dossier_modele" && zip -qr /content/modele_fr_rn.zip . && ls -lh /content/modele_fr_rn.zip
!cp /content/modele_fr_rn.zip $M/
print("Copié dans Google Drive :", f"{M}/modele_fr_rn.zip")

# Option : le mettre sur Hugging Face (privé), décommente et mets ton nom :
# from huggingface_hub import login; login()
# model.push_to_hub("NiBienvenu/modele-fr-rn", private=True); tok.push_to_hub("NiBienvenu/modele-fr-rn", private=True)